V0 du traitement de donnée

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import STL
import warnings
warnings.filterwarnings("ignore")

In [ ]:

FILE = "Rapports_Billing_demo,_2026-01-01_—_2026-06-23.csv"
# FILE = "Rapports_Billing_demo,_2026-01-01_—_2026-06-30.csv"

df = pd.read_csv(FILE, decimal=",")

# Conversion en datetime

df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df["Mois"] = df["Date"].dt.to_period("M").astype(str)
df["Jour"] = df["Date"].dt.to_period("D").astype(str)
# Garder uniquement la date (sans heure)

df["Date"] = df["Date"].dt.date
df = df[df["Coût catalogue (€)"]>1]

df[["Mois", "Date", "Coût catalogue (€)"]].head(10)

In [ ]:

fig = px.line(
    df,
    x="Jour",
    y="Sous-total (€)",
    color="Description du service",
    markers=True,
    title="Coûts GCP par service — Billing demo (2026)",
    labels={"Jour": "Jour", "Sous-total (€)": "Coût (€)", "Description du service": "Service"},
)

fig.update_layout(
    xaxis_title="Jour",
    yaxis_title="Coût (€)",
    legend_title="Service",
    hovermode="x unified",
)

fig.show()


## 1. Statistiques descriptives globales

In [ ]:
# Coût total par jour (toutes lignes agrégées)
daily = (
    df.groupby("Jour")["Sous-total (€)"]
    .sum()
    .sort_index()
    .rename("Total")
)
print(f"  Série journalière : {len(daily)} jours ({daily.index[0]} → {daily.index[-1]})")
daily

In [ ]:
# Pivot service × Jour (pour analyses par service)
pivot = (
    df.pivot_table(index="Jour", columns="Description du service",
                   values="Sous-total (€)", aggfunc="sum")
    .fillna(0)
    .sort_index()
)

s = daily.values.astype(float)
n = len(s)

mean_   = np.mean(s)
median_ = np.median(s)
std_    = np.std(s, ddof=1)
cv_     = std_ / mean_ * 100
skew_   = stats.skew(s)
kurt_   = stats.kurtosis(s, fisher=True)
q1, q3  = np.percentile(s, [25, 75])
iqr_    = q3 - q1
rng_    = s.max() - s.min()
mad_    = np.median(np.abs(s - median_))

print("══════════════════════════════════════════")
print("   STATISTIQUES DESCRIPTIVES — Coût Total")
print("══════════════════════════════════════════")
print(f"  N jours         : {n}")
print(f"  Somme totale    : {s.sum():>10.2f} €")
print(f"  Moyenne (μ)     : {mean_:>10.2f} €")
print(f"  Médiane         : {median_:>10.2f} €")
print(f"  Écart-type (σ)  : {std_:>10.2f} €")
print(f"  CV (σ/μ)        : {cv_:>9.1f} %")
print(f"  Min / Max       : {s.min():>7.2f} € / {s.max():.2f} €")
print(f"  Range           : {rng_:>10.2f} €")
print(f"  Q1 / Q3         : {q1:>7.2f} € / {q3:.2f} €")
print(f"  IQR             : {iqr_:>10.2f} €")
print(f"  MAD             : {mad_:>10.2f} €")
print(f"  Skewness        : {skew_:>10.4f}  {'(asymétrie droite)' if skew_>0 else '(asymétrie gauche)'}")
print(f"  Kurtosis (excès): {kurt_:>10.4f}  {'(leptokurtique)' if kurt_>0 else '(platykurtique)'}")
print("══════════════════════════════════════════")

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Boxplot journalier", "Distribution des coûts"])

fig.add_trace(go.Box(y=s, name="Coût total", boxpoints="all",
                     marker_color="steelblue", jitter=0.4, pointpos=-1.8), row=1, col=1)

fig.add_trace(go.Histogram(x=s, nbinsx=20, name="Distribution",
                           marker_color="steelblue", opacity=0.8), row=1, col=2)

for val, label, color in [(mean_, f"μ = {mean_:.0f}€", "red"),
                           (median_, f"Md = {median_:.0f}€", "orange")]:
    fig.add_vline(x=val, line_color=color, line_dash="dash",
                  annotation_text=label, row=1, col=2)

fig.update_layout(title="Distribution du coût journalier total", showlegend=False, height=420)
fig.show()

## 2. Tendance — Régression linéaire & moyenne mobile

In [ ]:
dates = daily.index

t = np.arange(n)
slope, intercept, r, p_val, se = stats.linregress(t, s)
trend_line = slope * t + intercept

# Intervalles de confiance 95%
t_crit = stats.t.ppf(0.975, df=n - 2)
s_err  = np.sqrt(np.sum((s - trend_line) ** 2) / (n - 2))
ci     = t_crit * s_err * np.sqrt(1/n + (t - t.mean())**2 / np.sum((t - t.mean())**2))

# Moyenne mobile 7 jours (centrée)
mm7 = pd.Series(s).rolling(7, center=True).mean().values

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=s, mode="lines+markers",
                         name="Coût réel", line=dict(color="steelblue", width=1.5),
                         marker=dict(size=3)))
fig.add_trace(go.Scatter(x=dates, y=trend_line, mode="lines",
                         name=f"Tendance linéaire (β={slope:+.2f}€/jour)",
                         line=dict(color="red", dash="dash", width=2)))
fig.add_trace(go.Scatter(
    x=np.concatenate([dates, dates[::-1]]),
    y=np.concatenate([trend_line + ci, (trend_line - ci)[::-1]]),
    fill="toself", fillcolor="rgba(255,0,0,0.1)",
    line=dict(color="rgba(255,255,255,0)"),
    name="IC 95% régression", showlegend=True))
fig.add_trace(go.Scatter(x=dates, y=mm7, mode="lines",
                         name="Moy. mobile 7 jours",
                         line=dict(color="orange", dash="dot", width=2)))

fig.update_layout(title=f"Tendance — β={slope:+.2f}€/jour | R²={r**2:.3f} | p={p_val:.4f}",
                  xaxis_title="Jour", yaxis_title="Coût total (€)",
                  hovermode="x unified", height=440)
fig.show()

print(f"\n  Pente (β)     : {slope:+.4f} €/jour")
print(f"  R²            : {r**2:.4f}  ({'tendance significative' if p_val < 0.05 else 'tendance non significative'}, p={p_val:.4f})")
print(f"  Projection J+1: {trend_line[-1] + slope:.2f} €")

## 3. Stationnarité — Tests ADF & KPSS

In [ ]:
# ── ADF (H0 : racine unitaire = non stationnaire) ────────────────────────────
adf_stat, adf_p, adf_lags, _, adf_cv, _ = adfuller(s, autolag="AIC")

# ── KPSS (H0 : stationnaire) ─────────────────────────────────────────────────
kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(s, regression="c", nlags="auto")

print("══════════════════════════════════════════════════")
print("   TEST ADF  (H₀ : non stationnaire / racine unit.)")
print("══════════════════════════════════════════════════")
print(f"  Statistique   : {adf_stat:.4f}")
print(f"  p-value       : {adf_p:.4f}  →  {'STATIONNAIRE ✓' if adf_p < 0.05 else 'NON STATIONNAIRE ✗'}")
for k, v in adf_cv.items():
    print(f"  Val. critique {k}: {v:.4f}")

print()
print("══════════════════════════════════════════════════")
print("   TEST KPSS (H₀ : stationnaire)")
print("══════════════════════════════════════════════════")
print(f"  Statistique   : {kpss_stat:.4f}")
print(f"  p-value       : {kpss_p:.4f}  →  {'NON STATIONNAIRE ✗' if kpss_p < 0.05 else 'STATIONNAIRE ✓'}")
for k, v in kpss_cv.items():
    print(f"  Val. critique {k}: {v:.4f}")

print()
adf_ok  = adf_p  < 0.05
kpss_ok = kpss_p >= 0.05
if adf_ok and kpss_ok:
    verdict = "Série STATIONNAIRE (ADF + KPSS concordent)"
elif not adf_ok and not kpss_ok:
    verdict = "Série NON STATIONNAIRE — différenciation recommandée"
else:
    verdict = "Résultats contradictoires — possible stationnarité de tendance (TS)"
print(f"  Verdict       : {verdict}")

# ── ACF / PACF — 20 lags pour capturer cycles hebdo et mensuels ──────────────
nlags = min(n - 2, 20)
acf_vals  = acf(s,  nlags=nlags, fft=True)
pacf_vals = pacf(s, nlags=nlags)
lags      = list(range(nlags + 1))
ci_bound  = 1.96 / np.sqrt(n)

fig = make_subplots(rows=1, cols=2, subplot_titles=["ACF", "PACF"])
for vals, col, name in [(acf_vals, 1, "ACF"), (pacf_vals, 2, "PACF")]:
    fig.add_trace(go.Bar(x=lags, y=vals, name=name,
                         marker_color=["red" if abs(v) > ci_bound else "steelblue"
                                       for v in vals]), row=1, col=col)
    for sign in [1, -1]:
        fig.add_hline(y=sign * ci_bound, line_dash="dash",
                      line_color="orange", row=1, col=col)

fig.update_layout(title="Auto-corrélations (ACF) et auto-corrélations partielles (PACF) — 20 lags",
                  showlegend=False, height=380)
fig.show()

## 4. Décomposition STL (Tendance / Saisonnalité / Résidus)

In [ ]:
# STL avec period=7 (saisonnalité hebdomadaire) sur données journalières
stl    = STL(daily, period=7, robust=True)
result = stl.fit()

stl_trend    = result.trend
stl_seasonal = result.seasonal
stl_resid    = result.resid

var_total    = np.var(s)
var_seasonal = np.var(stl_seasonal.values)
var_trend    = np.var(stl_trend.values)
var_resid    = np.var(stl_resid.values)
fs = max(0.0, 1 - var_resid / (var_seasonal + var_resid))
ft = max(0.0, 1 - var_resid / (var_trend    + var_resid))

print(f"  Force de saisonnalité (Fs) : {fs:.4f}  {'(forte)' if fs > 0.6 else '(modérée)' if fs > 0.3 else '(faible)'}")
print(f"  Force de tendance     (Ft) : {ft:.4f}  {'(forte)' if ft > 0.6 else '(modérée)' if ft > 0.3 else '(faible)'}")
print(f"  Variance résiduelle        : {var_resid:.2f} €²  ({var_resid/var_total*100:.1f}% du total)")

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=["Série originale", "Tendance STL",
                                    f"Saisonnalité hebdo (Fs={fs:.2f})", "Résidus"])

colors = ["steelblue", "red", "green", "grey"]
series = [s, stl_trend.values, stl_seasonal.values, stl_resid.values]
modes  = ["lines", "lines", "lines", "lines+markers"]

for i, (vals, color, mode) in enumerate(zip(series, colors, modes), start=1):
    fig.add_trace(go.Scatter(x=dates, y=vals, mode=mode,
                             line=dict(color=color, width=1.5),
                             marker=dict(size=3)), row=i, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="black", line_width=1, row=4, col=1)

fig.update_layout(title="Décomposition STL — Tendance / Saisonnalité hebdo / Résidus",
                  showlegend=False, height=700)
fig.show()

## 5. Détection d'anomalies — Z-score & IQR

In [ ]:
z_scores   = np.abs(stats.zscore(s))
iqr_lower  = q1 - 1.5 * iqr_
iqr_upper  = q3 + 1.5 * iqr_

anomalies_z   = daily[z_scores > 2].index.tolist()
anomalies_iqr = daily[(s < iqr_lower) | (s > iqr_upper)].index.tolist()
all_anomalies = sorted(set(anomalies_z) | set(anomalies_iqr))

print(f"  Anomalies Z-score > 2  : {len(anomalies_z)} jour(s) : {anomalies_z or 'aucune'}")
print(f"  Anomalies IQR fence    : {len(anomalies_iqr)} jour(s) : {anomalies_iqr or 'aucune'}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=dates, y=s, mode="lines",
                         name="Coût réel", line=dict(color="steelblue", width=1.5)))

fig.add_hrect(y0=mean_ - 2*std_, y1=mean_ + 2*std_,
              fillcolor="rgba(255,165,0,0.12)", line_width=0,
              annotation_text="±2σ", annotation_position="top left")
fig.add_hline(y=mean_, line_dash="dash", line_color="red",
              annotation_text=f"μ = {mean_:.0f}€")

if all_anomalies:
    anom_vals = [daily[d] for d in all_anomalies]
    fig.add_trace(go.Scatter(x=all_anomalies, y=anom_vals,
                             mode="markers", name="Anomalie",
                             marker=dict(color="red", size=10, symbol="x")))

fig.update_layout(title="Détection d'anomalies — bandes ±2σ & IQR fence",
                  xaxis_title="Jour", yaxis_title="Coût (€)",
                  hovermode="x unified", height=420)
fig.show()

# Heatmap de variation WoW (semaine sur semaine) par service
pivot_weekly = pivot.copy()
pivot_weekly.index = pd.to_datetime(pivot_weekly.index)
pivot_weekly = pivot_weekly.resample("W").sum()
wow = pivot_weekly.pct_change() * 100
wow_clean = wow.dropna().replace([np.inf, -np.inf], np.nan).fillna(0)

fig2 = px.imshow(wow_clean.T,
                 color_continuous_scale="RdYlGn",
                 color_continuous_midpoint=0,
                 title="Variation WoW (%) par service — Heatmap",
                 labels=dict(x="Semaine", y="Service", color="Δ%"),
                 aspect="auto", height=500)
fig2.show()

## 6. Pareto & Volatilité par service

In [ ]:
svc_rows = []
for svc in pivot.columns:
    vals        = pivot[svc].values
    active_vals = vals[vals > 0]
    if len(active_vals) == 0:
        continue
    svc_rows.append({
        "Service":      svc,
        "Total (€)":    float(np.sum(vals)),
        "Moy. (€)":     float(np.mean(active_vals)),
        "Méd. (€)":     float(np.median(active_vals)),
        "Std (€)":      float(np.std(active_vals, ddof=1)) if len(active_vals) > 1 else 0.0,
        "CV (%)":       float(np.std(active_vals, ddof=1) / np.mean(active_vals) * 100)
                        if len(active_vals) > 1 and np.mean(active_vals) > 0 else 0.0,
        "Jours actifs": int(np.sum(vals > 0)),
    })

svc_df = pd.DataFrame(svc_rows).sort_values("Total (€)", ascending=False).reset_index(drop=True)
svc_df["Part (%)"]   = svc_df["Total (€)"] / svc_df["Total (€)"].sum() * 100
svc_df["Cumulé (%)"] = svc_df["Part (%)"].cumsum()

print(f"{'Service':<28} {'Total':>8}  {'Moy':>7}  {'Méd':>7}  {'Std':>7}  {'CV':>6}  {'Jours':>5}  {'Part':>5}  {'Cumulé':>6}")
print("-" * 95)
for _, row in svc_df.iterrows():
    marker = " ← 80%" if (row["Cumulé (%)"] >= 80 and (row["Cumulé (%)"] - row["Part (%)"]) < 80) else ""
    print(f"{row['Service']:<28} {row['Total (€)']:>8.2f}  {row['Moy. (€)']:>7.2f}  "
          f"{row['Méd. (€)']:>7.2f}  {row['Std (€)']:>7.2f}  {row['CV (%)']:>5.1f}%  "
          f"{row['Jours actifs']:>5}  {row['Part (%)']:>5.1f}%  {row['Cumulé (%)']:>6.1f}%{marker}")

top80 = svc_df.loc[svc_df["Cumulé (%)"].shift(1, fill_value=0) < 80, "Service"].tolist()
print(f"\n  Règle 80/20 → {len(top80)} service(s) représentent 80% du coût : {', '.join(top80)}")

# Pareto chart
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=svc_df["Service"], y=svc_df["Part (%)"],
                     marker_color="steelblue", opacity=0.85, name="Part (%)",
                     text=[f"{v:.1f}%" for v in svc_df["Part (%)"]],
                     textposition="outside"), secondary_y=False)
fig.add_trace(go.Scatter(x=svc_df["Service"], y=svc_df["Cumulé (%)"],
                         mode="lines+markers", name="Cumulé (%)",
                         line=dict(color="red", width=2)), secondary_y=True)
fig.add_trace(go.Scatter(x=svc_df["Service"], y=[80]*len(svc_df),
                         mode="lines", name="Seuil 80%",
                         line=dict(color="orange", dash="dash", width=1.5)), secondary_y=True)
fig.update_yaxes(title_text="Part individuelle (%)", secondary_y=False)
fig.update_yaxes(title_text="Cumulé (%)", range=[0, 115], secondary_y=True)
fig.update_layout(title="Diagramme de Pareto — Concentration des coûts par service",
                  xaxis_tickangle=-30, height=480)
fig.show()

# Volatilité CV
cv_df = svc_df[svc_df["CV (%)"] > 0].sort_values("CV (%)", ascending=False)
fig2  = px.bar(cv_df, x="Service", y="CV (%)",
               title="Volatilité par service — Coefficient de Variation (σ/μ × 100)",
               color="CV (%)", color_continuous_scale="Reds",
               text=[f"{v:.1f}%" for v in cv_df["CV (%)"]],
               height=420)
fig2.update_traces(textposition="outside")
fig2.update_layout(xaxis_tickangle=-30, coloraxis_showscale=False)
fig2.show()

## Export pour le notebook de benchmark

In [ ]:
# Export pour Benchmark_Forecasting.ipynb
daily_export = daily.reset_index()
daily_export.columns = ["ds", "y"]
daily_export["ds"] = pd.to_datetime(daily_export["ds"])
daily_export.to_parquet("daily_costs.parquet", index=False)

pivot_export = pivot.copy()
pivot_export.index = pd.to_datetime(pivot_export.index)
pivot_export.index.name = "ds"
pivot_export.reset_index().to_parquet("daily_per_service.parquet", index=False)

print(f"  daily_costs.parquet        — {len(daily_export)} lignes, colonnes : {list(daily_export.columns)}")
print(f"  daily_per_service.parquet  — {len(pivot_export)} jours × {len(pivot_export.columns)} services")